In [1]:
#Jeffrey Tam
#INST 414
#October 23, 2026
#Module 3

In [2]:
%matplotlib inline

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.sparse import lil_matrix

#Used to deal with memory leak warnings from sklearn lib
import warnings
warnings.simplefilter('ignore')

In [4]:
file_name = "openings.csv"

In [ ]:
#Subset of original dataframe
file_name = "openings.csv"
df = pd.read_csv(file_name)
#Cleaning dataset
df = df.fillna(0)

#Note, Unnamed: 0 is the opening id
new_df = df[["Unnamed: 0", "Opening", "move1w", "move1b", "move2w", "move2b", "move3w", "move3b", "move4w", "move4b"]].copy()
new_df

In [ ]:
#Mapping opening id to the opening name
list_opening_id = new_df["Unnamed: 0"].values.tolist()
list_opening_name = new_df["Opening"].values.tolist()


opening_to_id_map = dict(zip(list_opening_id, list_opening_name))
id_to_moves = {} #Used to id the moves, as a test, since every name is different, but switch it to ID for clearity/efficency 
move_id_to_moves = {} #Used to identify move sets in a map for matrix

for r in new_df.itertuples(index=False):
    #Used to clean up 1st move on white side, returns scientific notation instead of a string, i.e e3 = 00E+03
    move1w = r.move1w
    # if(r.move1w == "00E+03" or r.move1w == "00E+04"):
    #     print({r.Opening: [r.move1w, r.move1b, r.move2w, r.move2b, r.move3w, r.move3b, r.move4w, r.move4b]})
    if(move1w == "00E+03"):
        move1w = "e3"
    if(move1w == "00E+04"):
        move1w = "e4"
    id_to_moves.update({r._0: [move1w, r.move1b, r.move2w, r.move2b, r.move3w, r.move3b, r.move4w, r.move4b]})

# for k, v in id_to_moves.items():
#     print(k, " ", v)

In [ ]:
#Columns for every move by opening from 8 columns
positions_set= set()
posit_names = ['move1w', 'move1b', 'move2w', 'move2b', 'move3w','move3b', 'move4w', 'move4b']
for name in posit_names:
    list_a = new_df[name].values.tolist()
    positions_set.update(list_a)

positions_set.discard(0)
positions_set.discard('00E+03')
positions_set.discard('00E+04')
#Num of features in the matrix
len(positions_set)

count = 0
#Convert set to a map
for x in positions_set:
    move_id_to_moves[x] = count
    count += 1

In [ ]:
#Sparse matrix
matrix_sparse = lil_matrix((len(list_opening_name), len(positions_set)), dtype=bool)

for row, opening in enumerate(list_opening_name):
    for move in id_to_moves[row]:
        if(move != 0):
            this_index = move_id_to_moves[move]
            matrix_sparse[row, this_index] = 1


In [ ]:
#new data frame 
df = pd.DataFrame.sparse.from_spmatrix(
    matrix_sparse,
    index=list_opening_name,
    columns=positions_set
)

df = df.astype(int)
df

In [ ]:
#Using Elbow Method to find best num of cluster
interia_scores = []
for test_k in range(2, 60, 1):
    tmp_model = KMeans(n_clusters=test_k, random_state=42)
    tmp_model.fit(df)
    
    score = tmp_model.inertia_
    #internia score is divided by 1000 for readablity
    interia_scores.append((test_k, score/1000))

intertia_df = pd.DataFrame(interia_scores, columns=["k", "score"])  
intertia_df

In [ ]:
#Based on KMeans 35 for k because 
fig = plt.figure(figsize=(16,9))
axis = fig.add_subplot(1, 1, 1)
intertia_df.sort_values(by="k").plot("k", "score", ax=axis)

axis.set_ylabel("Intertia / 1000")
plt.show()

In [ ]:
#Finding Best K value with elbow method 
k = 21
cluster_model = KMeans(n_clusters=k, random_state=42)
cluster_model.fit(df)

In [ ]:
#Make cluster using Kmeans/Edclidian 
cluster_labels = cluster_model.predict(df)
openings_cluster_df = pd.DataFrame(cluster_labels, index=df.index, columns=["clusters"])
openings_cluster_df["clusters"].value_counts()

In [ ]:
for cluster, openings in openings_cluster_df.groupby("clusters"):
    print(f"cluster: {cluster}, size: {len(openings)}")
    for opening in openings.sample(4).index:
        print(opening)
    print()